# Kabadiwala Connect — E-waste Classification & Price Estimation

Three-stage pipeline, trained on the datasets produced by `dataset_pipeline.py` and
emitting payloads that conform to the **Kabadiwala Connect Data Dictionary** (`Lot` entity).

| Stage | Model | Output |
|---|---|---|
| **Pipeline 1** | ResNet18 binary classifier (YOLOv8-cls optional) | `is_ewaste`, `confidence` |
| **Pipeline 2** | ResNet34 multi-task: 1 trunk, 3 heads | `subCategory`, component flags, metal fractions |
| **Pipeline 3** | XGBoost regressor | `estimatedValue` (₹) |

**Run order:** cells 1 → 7, top to bottom. Set `KC_PIPELINE_DIR` if this notebook does not
sit next to `dataset_pipeline.py`.

### Three things to know before trusting any number this produces

1. **`p1_items` / `p2_items` are not importable.** `dataset_pipeline.py` builds them inside
   `if __name__ == "__main__":`. Cell 2 defines `build_pipeline_items()`, which replays the
   same calls in the same order. Hoisting that block into a function in `dataset_pipeline.py`
   would let the notebook drop the shim.
2. **Two category vocabularies exist.** `map_class_to_category()` emits `CircuitBoard`,
   `CableWire`, `ScrapMetal`…; the Data Dictionary specifies `PCB`, `Cables`, `Motor`, `CRT`,
   `Battery`. `PIPELINE_TO_LOT` in Cell 2 is the single translation point. `CRT`, `Battery`
   and `Motor` have **no source data** — the model cannot predict them yet.
3. **Component, metal-content and price labels are synthetic.** No source dataset annotates
   internal components or composition, and there is no price ledger. Cell 2 uses documented
   teardown priors as weak labels; Cell 5 simulates `estimatedValue` from scrap rates.
   Treat Cell 7's price metrics as a plumbing check, not as accuracy.

## Cell 1 — Dependencies & Setup

In [ ]:
# =====================================================================
# CELL 1 - Dependencies, environment check, reproducibility
# =====================================================================
# On a fresh machine, run this once (then restart the kernel):
# !pip install torch torchvision ultralytics xgboost scikit-learn \
#              opencv-python pillow pandas numpy matplotlib pyyaml tqdm \
#              roboflow joblib

from __future__ import annotations

import json
import math
import os
import random
import sys
import uuid
import warnings
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms

warnings.filterwarnings("ignore", category=UserWarning)

# --- Reproducibility -------------------------------------------------
SEED = 42


def set_seed(seed: int = SEED) -> None:
    """Pin every stochastic component we touch to a single seed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(SEED)

# --- Device ----------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 0 if os.name == "nt" else min(4, (os.cpu_count() or 2))


def make_scaler():
    """GradScaler across torch versions; None when AMP is off."""
    if not USE_AMP:
        return None
    try:
        return torch.amp.GradScaler(DEVICE.type)          # torch >= 2.4
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()                # older torch

# --- Optional dependencies (the notebook degrades gracefully) --------
try:
    import xgboost as xgb

    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from ultralytics import YOLO

    HAS_ULTRALYTICS = True
except ImportError:
    HAS_ULTRALYTICS = False

try:
    import cv2

    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False

try:
    import joblib

    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("=" * 62)
print("Kabadiwala Connect - e-waste classification & price estimation")
print("=" * 62)
print(f"python        : {sys.version.split()[0]}")
print(f"torch         : {torch.__version__}")
print(f"torchvision   : {torchvision.__version__}")
print(f"device        : {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"mixed precision: {USE_AMP}")
print(f"dataloader workers: {NUM_WORKERS}")
print(f"xgboost       : {'yes' if HAS_XGBOOST else 'NO  -> Cell 5 falls back to sklearn'}")
print(f"ultralytics   : {'yes' if HAS_ULTRALYTICS else 'NO  -> YOLO paths disabled, CNN used'}")
print(f"opencv        : {'yes' if HAS_CV2 else 'no  (PIL is used for all I/O anyway)'}")
print(f"seed          : {SEED}")

## Cell 2 — Dataset Loading & Preparation

Imports `dataset_pipeline.py`, normalises both item lists into `Lot`-schema DataFrames,
and builds train/val/test DataLoaders. If no source images are on disk, a clearly-labelled
demo dataset is synthesised so the notebook stays runnable end-to-end.

In [ ]:
# =====================================================================
# CELL 2 - Dataset ingestion, Lot-schema standardisation, DataLoaders
# =====================================================================

# ---------------------------------------------------------------------
# 2.1  Import dataset_pipeline.py
# ---------------------------------------------------------------------
# Point KC_PIPELINE_DIR at the folder holding dataset_pipeline.py if the
# notebook does not live next to it.
DATASET_PIPELINE_DIR = Path(os.environ.get("KC_PIPELINE_DIR", ".")).resolve()
if not (DATASET_PIPELINE_DIR / "dataset_pipeline.py").exists():
    raise FileNotFoundError(
        f"dataset_pipeline.py not found in {DATASET_PIPELINE_DIR}. "
        "Set the KC_PIPELINE_DIR environment variable to its folder."
    )
if str(DATASET_PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(DATASET_PIPELINE_DIR))

import dataset_pipeline as dp  # noqa: E402
from dataset_pipeline import (  # noqa: E402
    WORK_DIR,
    RAW_DIR,
    CROP_DIR,
    CKPT_DIR,
    TRASHNET_IMAGES,
    ROBOFLOW_SOURCES,
    IMG_EXTS,
    download_roboflow_datasets,
    gather_yolo_images,
    crop_yolo_export,
    index_class_folders,
    finalize_splits,
    map_class_to_category,
)

ARTIFACT_DIR = WORK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def build_pipeline_items(run_download: bool = True) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Rebuild dataset_pipeline's `p1_items` / `p2_items` as an importable call.

    NOTE: `from dataset_pipeline import p1_items` raises ImportError, because
    those two lists are only created inside `if __name__ == "__main__":`.
    This function replays the exact same sequence of calls so the notebook
    stays a thin consumer of the pipeline rather than a fork of it.
    """
    if run_download:
        download_roboflow_datasets()

    p1_items: List[Dict[str, Any]] = []
    p2_items: List[Dict[str, Any]] = []

    rf_exports = [RAW_DIR / f"{s['project']}_v{s['version']}" for s in ROBOFLOW_SOURCES]
    for export in rf_exports:
        if export.exists():
            print(f"Processing dataset: {export.name}")
            p1_items += [{**it, "label": 1} for it in gather_yolo_images(export)]
            p2_items += crop_yolo_export(export, CROP_DIR)
        else:
            print(f"[Missing] Directory not found: {export.resolve()}")

    if TRASHNET_IMAGES.exists():
        print(f"Processing TrashNet plastics from: {TRASHNET_IMAGES.name}")
        plastics = index_class_folders(
            TRASHNET_IMAGES, lambda n: "MixedPlastics" if n == "plastic" else None
        )
        for it in plastics:
            p1_items.append({**it, "label": 0})
            p2_items.append(it)
    else:
        print(f"[Missing] TrashNet folder not found: {TRASHNET_IMAGES.resolve()}")

    finalize_splits(p1_items)
    finalize_splits(p2_items)
    return p1_items, p2_items


# ---------------------------------------------------------------------
# 2.2  Data Dictionary constants (single source of truth)
# ---------------------------------------------------------------------
# These mirror the Kabadiwala Connect Data Dictionary verbatim. Field names
# are camelCase, exactly as the Dart/Flutter layer expects. Money is a raw
# double in INR, never a formatted string.
LOT_CATEGORIES: List[str] = [
    "PCB", "CRT", "Cables", "Battery", "Motor", "MixedPlastics", "OtherEwaste",
]
SYNC_STATUSES: List[str] = ["pending", "synced", "failed"]

# subCategory vocabulary the AI classifier is allowed to emit, plus the
# category each one rolls up into.
SUBCATEGORY_TO_CATEGORY: Dict[str, str] = {
    "Motherboard": "PCB",
    "MobilePhone": "OtherEwaste",
    "Laptop": "OtherEwaste",
    "Keyboard": "OtherEwaste",
    "Mouse": "OtherEwaste",
    "Cable": "Cables",
    "ScrapMetal": "OtherEwaste",
}
CANONICAL_SUBCATEGORIES: List[str] = list(SUBCATEGORY_TO_CATEGORY.keys())

# SCHEMA BRIDGE - dataset_pipeline.map_class_to_category() emits its own
# vocabulary ("CircuitBoard", "MobilePhone", "CableWire", "ScrapMetal", ...)
# which is NOT the Data Dictionary vocabulary. Everything downstream of this
# dict speaks Data Dictionary only. Add a row here when new source labels
# appear; do not special-case categories anywhere else in the notebook.
PIPELINE_TO_LOT: Dict[str, Tuple[str, Optional[str]]] = {
    "CircuitBoard":  ("PCB",           "Motherboard"),
    "MobilePhone":   ("OtherEwaste",   "MobilePhone"),
    "CableWire":     ("Cables",        "Cable"),
    "MixedPlastics": ("MixedPlastics", None),
    "ScrapMetal":    ("OtherEwaste",   "ScrapMetal"),
    "OtherEwaste":   ("OtherEwaste",   None),
}
# Coverage gap worth knowing about: no current source labels CRT, Battery or
# Motor, so those three categories are reachable from the UI but the model
# will never predict them until labelled data exists.

COMPONENT_KEYS: List[str] = ["has_motor", "has_cables", "has_pcb", "has_battery", "has_display"]
METAL_KEYS: List[str] = ["pct_copper", "pct_aluminum", "pct_steel_iron", "pct_precious_pcb"]

# ---------------------------------------------------------------------
# 2.3  Weak-label priors for Pipeline 2
# ---------------------------------------------------------------------
# None of the source datasets carry component or metal-content annotations.
# These tables are domain priors (teardown averages), used as *weak labels*
# so the multi-task heads have a target to regress toward. They are not
# ground truth - replace them with measured values from real lots as soon as
# the recycler side starts reporting composition back.
COMPONENT_PRIORS: Dict[str, Dict[str, float]] = {
    "Motherboard": dict(has_motor=0.02, has_cables=0.30, has_pcb=1.00, has_battery=0.20, has_display=0.00),
    "MobilePhone": dict(has_motor=0.35, has_cables=0.05, has_pcb=1.00, has_battery=0.95, has_display=0.95),
    "Laptop":      dict(has_motor=0.70, has_cables=0.60, has_pcb=1.00, has_battery=0.90, has_display=0.95),
    "Keyboard":    dict(has_motor=0.02, has_cables=0.85, has_pcb=0.60, has_battery=0.10, has_display=0.02),
    "Mouse":       dict(has_motor=0.15, has_cables=0.70, has_pcb=0.80, has_battery=0.25, has_display=0.00),
    "Cable":       dict(has_motor=0.00, has_cables=1.00, has_pcb=0.05, has_battery=0.00, has_display=0.00),
    "ScrapMetal":  dict(has_motor=0.20, has_cables=0.10, has_pcb=0.05, has_battery=0.02, has_display=0.00),
}
CATEGORY_COMPONENT_FALLBACK: Dict[str, Dict[str, float]] = {
    "MixedPlastics": dict(has_motor=0.00, has_cables=0.00, has_pcb=0.00, has_battery=0.00, has_display=0.00),
    "OtherEwaste":   dict(has_motor=0.20, has_cables=0.30, has_pcb=0.45, has_battery=0.25, has_display=0.20),
}

# Mass fraction of the whole lot, per metal bucket. They sum to <= 1.0; the
# remainder is plastic/glass/board and carries no metal value.
METAL_PRIORS: Dict[str, Dict[str, float]] = {
    "Motherboard": dict(pct_copper=0.12, pct_aluminum=0.05, pct_steel_iron=0.08, pct_precious_pcb=0.040),
    "MobilePhone": dict(pct_copper=0.08, pct_aluminum=0.12, pct_steel_iron=0.05, pct_precious_pcb=0.030),
    "Laptop":      dict(pct_copper=0.07, pct_aluminum=0.20, pct_steel_iron=0.10, pct_precious_pcb=0.015),
    "Keyboard":    dict(pct_copper=0.02, pct_aluminum=0.01, pct_steel_iron=0.05, pct_precious_pcb=0.005),
    "Mouse":       dict(pct_copper=0.03, pct_aluminum=0.01, pct_steel_iron=0.03, pct_precious_pcb=0.005),
    "Cable":       dict(pct_copper=0.45, pct_aluminum=0.05, pct_steel_iron=0.01, pct_precious_pcb=0.000),
    "ScrapMetal":  dict(pct_copper=0.10, pct_aluminum=0.25, pct_steel_iron=0.55, pct_precious_pcb=0.000),
}
CATEGORY_METAL_FALLBACK: Dict[str, Dict[str, float]] = {
    "MixedPlastics": dict(pct_copper=0.00, pct_aluminum=0.00, pct_steel_iron=0.00, pct_precious_pcb=0.000),
    "OtherEwaste":   dict(pct_copper=0.05, pct_aluminum=0.08, pct_steel_iron=0.15, pct_precious_pcb=0.010),
}


def component_prior(category: str, sub_category: Optional[str]) -> Dict[str, float]:
    if sub_category in COMPONENT_PRIORS:
        return COMPONENT_PRIORS[sub_category]
    return CATEGORY_COMPONENT_FALLBACK.get(category, CATEGORY_COMPONENT_FALLBACK["OtherEwaste"])


def metal_prior(category: str, sub_category: Optional[str]) -> Dict[str, float]:
    if sub_category in METAL_PRIORS:
        return METAL_PRIORS[sub_category]
    return CATEGORY_METAL_FALLBACK.get(category, CATEGORY_METAL_FALLBACK["OtherEwaste"])


# ---------------------------------------------------------------------
# 2.4  Lot entity
# ---------------------------------------------------------------------
@dataclass
class Lot:
    """One collected material item/batch - mirrors the Data Dictionary."""

    id: str
    category: str
    approxWeightKg: float
    photoPaths: List[str] = field(default_factory=list)
    subCategory: Optional[str] = None
    estimatedValue: Optional[float] = None
    quotedPrice: Optional[float] = None
    finalSaleValue: Optional[float] = None
    createdAt: Optional[str] = None
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    syncStatus: str = "pending"
    recyclerId: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def validate_lot(lot: Dict[str, Any]) -> Dict[str, Any]:
    """Fail loudly if a payload drifts from the Data Dictionary contract."""
    if lot["category"] not in LOT_CATEGORIES:
        raise ValueError(f"category {lot['category']!r} not in {LOT_CATEGORIES}")
    if lot["subCategory"] is not None and lot["subCategory"] not in CANONICAL_SUBCATEGORIES:
        raise ValueError(f"subCategory {lot['subCategory']!r} not in {CANONICAL_SUBCATEGORIES}")
    if lot["syncStatus"] not in SYNC_STATUSES:
        raise ValueError(f"syncStatus {lot['syncStatus']!r} not in {SYNC_STATUSES}")
    if not isinstance(lot["id"], str):
        raise TypeError("id must be String (uuid) - IDs are always String across the API boundary")
    if not isinstance(lot["approxWeightKg"], float):
        raise TypeError("approxWeightKg must be double")
    if not isinstance(lot["photoPaths"], list):
        raise TypeError("photoPaths must be List<String>")
    for money in ("estimatedValue", "quotedPrice", "finalSaleValue"):
        if lot[money] is not None and not isinstance(lot[money], float):
            raise TypeError(f"{money} must be double? in INR, never a formatted string")
    return lot


# ---------------------------------------------------------------------
# 2.5  Demo fallback (only fires when no real data is on disk)
# ---------------------------------------------------------------------
def synthesize_demo_dataset(n_per_class: int = 60) -> Tuple[List[Dict], List[Dict]]:
    """Generate tiny procedural images so the notebook is runnable end-to-end.

    This exists purely to smoke-test the plumbing when the Roboflow/TrashNet
    downloads have not run. Metrics produced from it are meaningless.
    """
    print("\n[DEMO MODE] No source images found - generating synthetic stand-ins.")
    demo_root = WORK_DIR / "demo"
    rng = np.random.default_rng(SEED)
    palette = {
        "CircuitBoard": (30, 110, 60), "MobilePhone": (40, 40, 45),
        "CableWire": (160, 90, 40), "ScrapMetal": (150, 150, 160),
        "MixedPlastics": (60, 120, 200),
    }
    p1_items, p2_items = [], []
    for cat, base in palette.items():
        for i in range(n_per_class):
            split = "train" if i < int(0.8 * n_per_class) else ("val" if i < int(0.9 * n_per_class) else "test")
            out = demo_root / split / cat
            out.mkdir(parents=True, exist_ok=True)
            path = out / f"{cat}_{i:04d}.jpg"
            if not path.exists():
                noise = rng.normal(0, 28, (96, 96, 3))
                arr = np.clip(np.array(base, dtype=float) + noise, 0, 255).astype(np.uint8)
                Image.fromarray(arr).save(path, quality=88)
            item = {"image_path": str(path), "category": cat, "split": split, "source": "demo"}
            p1_items.append({**item, "label": 0 if cat == "MixedPlastics" else 1})
            p2_items.append(item)
    return p1_items, p2_items


# ---------------------------------------------------------------------
# 2.6  Items -> DataFrames in Lot vocabulary
# ---------------------------------------------------------------------
def items_to_dataframe(items: List[Dict[str, Any]], kind: str) -> pd.DataFrame:
    """Normalise raw pipeline items into a Lot-schema DataFrame.

    kind="p1" keeps the binary `label`; kind="p2" adds category/subCategory
    plus the weak component and metal targets.
    """
    rows: List[Dict[str, Any]] = []
    missing = 0
    for it in items:
        path = Path(it["image_path"])
        if not path.exists():
            missing += 1
            continue
        pipeline_cat = it.get("category")
        category, sub_category = PIPELINE_TO_LOT.get(pipeline_cat, (None, None)) if pipeline_cat else (None, None)
        row: Dict[str, Any] = {
            "id": str(uuid.uuid4()),
            "image_path": str(path),
            "photoPaths": [str(path)],
            "split": it.get("split", "train"),
            "source": it.get("source", "unknown"),
            "pipelineCategory": pipeline_cat,
            "category": category,
            "subCategory": sub_category,
            "syncStatus": "pending",
        }
        if kind == "p1":
            row["label"] = int(it["label"])
        else:
            if category is None:
                continue  # unmapped source label - skip rather than guess
            row.update(component_prior(category, sub_category))
            row.update(metal_prior(category, sub_category))
        rows.append(row)

    df = pd.DataFrame(rows)
    if missing:
        print(f"[warn] {missing} item(s) referenced a path that no longer exists - dropped.")
    return df


# ---------------------------------------------------------------------
# 2.7  Torch datasets
# ---------------------------------------------------------------------
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image(path: str) -> Image.Image:
    """Open an image as RGB, raising a uniform error type on any failure."""
    try:
        with Image.open(path) as im:
            return im.convert("RGB")
    except (FileNotFoundError, OSError, UnidentifiedImageError) as exc:
        raise FileNotFoundError(f"Unreadable image: {path} ({exc})") from exc


class LotImageDataset(Dataset):
    """Serves either the binary task ("p1") or the multi-task head set ("p2").

    A corrupt or vanished file yields a zero tensor with ok=0 so a single bad
    file never kills a training run; `ok` is used to mask the loss.
    """

    def __init__(self, df: pd.DataFrame, transform, mode: str = "p1"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.mode = mode

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        ok = 1.0
        try:
            image = self.transform(load_image(row["image_path"]))
        except FileNotFoundError:
            image = torch.zeros(3, IMG_SIZE, IMG_SIZE)
            ok = 0.0

        sample: Dict[str, torch.Tensor] = {"image": image, "ok": torch.tensor(ok)}
        if self.mode == "p1":
            sample["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        else:
            sub = row["subCategory"]
            # -100 = ignore_index: rows with no subCategory still train the
            # component and metal heads, they just skip the CE term.
            sample["y_sub"] = torch.tensor(
                CANONICAL_SUBCATEGORIES.index(sub) if sub in CANONICAL_SUBCATEGORIES else -100,
                dtype=torch.long,
            )
            sample["y_comp"] = torch.tensor([float(row[k]) for k in COMPONENT_KEYS], dtype=torch.float32)
            sample["y_metal"] = torch.tensor([float(row[k]) for k in METAL_KEYS], dtype=torch.float32)
        return sample


def make_loaders(df: pd.DataFrame, mode: str, batch_size: int = 32,
                 balance_train: bool = True) -> Dict[str, DataLoader]:
    """Build train/val/test loaders, class-balancing the training sampler."""
    loaders: Dict[str, DataLoader] = {}
    for split in ("train", "val", "test"):
        part = df[df["split"] == split]
        if part.empty:
            continue
        is_train = split == "train"
        ds = LotImageDataset(part, train_tf if is_train else eval_tf, mode=mode)

        sampler = None
        if is_train and balance_train:
            key = "label" if mode == "p1" else "category"
            counts = part[key].value_counts().to_dict()
            weights = [1.0 / counts[v] for v in part[key]]
            sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

        loaders[split] = DataLoader(
            ds, batch_size=batch_size, sampler=sampler, shuffle=(is_train and sampler is None),
            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"), drop_last=False,
        )
    return loaders


# ---------------------------------------------------------------------
# 2.8  Run ingestion
# ---------------------------------------------------------------------
RUN_ROBOFLOW_DOWNLOAD = True   # set False on an air-gapped box
P2_EXCLUDE_PLASTICS = True     # Pipeline 2 only ever sees e-waste at inference

p1_items, p2_items = build_pipeline_items(run_download=RUN_ROBOFLOW_DOWNLOAD)
DEMO_MODE = len(p1_items) == 0
if DEMO_MODE:
    p1_items, p2_items = synthesize_demo_dataset()

p1_df = items_to_dataframe(p1_items, kind="p1")
p2_df = items_to_dataframe(p2_items, kind="p2")
if P2_EXCLUDE_PLASTICS:
    p2_df = p2_df[p2_df["category"] != "MixedPlastics"].reset_index(drop=True)

if p1_df.empty or p2_df.empty:
    raise RuntimeError("No usable images after normalisation - check RAW_DIR and CROP_DIR.")

p1_loaders = make_loaders(p1_df, mode="p1", batch_size=32)
p2_loaders = make_loaders(p2_df, mode="p2", batch_size=32)

print("\n--- Pipeline 1 (binary) ---")
print(p1_df.groupby(["split", "label"]).size().unstack(fill_value=0))
print("\n--- Pipeline 2 (hierarchical) ---")
print(p2_df.groupby(["split", "category"]).size().unstack(fill_value=0))
print("\nsubCategory distribution:")
print(p2_df["subCategory"].fillna("<none>").value_counts())
print(f"\nsource breakdown: {p2_df['source'].value_counts().to_dict()}")
if DEMO_MODE:
    print("\n*** DEMO MODE - synthetic images. Metrics below are not meaningful. ***")

## Cell 3 — Pipeline 1: Binary Classifier (E-waste vs. MixedPlastics)

In [ ]:
# =====================================================================
# CELL 3 - Pipeline 1: binary classifier (E-waste vs Mixed Plastics)
# =====================================================================
# label 1 = e-waste, label 0 = MixedPlastics.
# ResNet18 transfer learning. A YOLOv8-cls variant is wired in behind
# USE_YOLO_CLS for teams that prefer the ultralytics training loop; the
# torch path is the default because it keeps one runtime for all 3 heads.

USE_YOLO_CLS = False
P1_EPOCHS = 8
P1_LR = 3e-4
P1_CKPT = CKPT_DIR / "pipeline1_binary.pt"


def build_backbone(arch: str = "resnet18", pretrained: bool = True) -> Tuple[nn.Module, int]:
    """Return an ImageNet backbone with the classifier stripped off.

    Falls back to random init if the weight download is unavailable (offline
    box, proxy, etc.) rather than crashing the run.
    """
    factory = getattr(torchvision.models, arch)
    model = None
    if pretrained:
        try:
            model = factory(weights="DEFAULT")
        except Exception as exc:  # no network / no cached weights
            print(f"[warn] pretrained weights unavailable ({exc}); training from scratch.")
    if model is None:
        model = factory(weights=None)
    feat_dim = model.fc.in_features
    model.fc = nn.Identity()
    return model, feat_dim


class BinaryEwasteNet(nn.Module):
    """Backbone + 2-way head. Two logits (not one) keeps the head shape
    identical to the multi-class heads in Cell 4."""

    def __init__(self, arch: str = "resnet18", pretrained: bool = True, dropout: float = 0.2):
        super().__init__()
        self.backbone, feat_dim = build_backbone(arch, pretrained)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 2))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))


def run_epoch_p1(model, loader, criterion, optimizer=None, scaler=None) -> Tuple[float, float]:
    """One pass. optimizer=None -> evaluation mode."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, seen = 0.0, 0, 0

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        mask = batch["ok"].to(DEVICE) > 0
        if mask.sum() == 0:
            continue
        images, labels = images[mask], labels[mask]

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
        seen += bs

    return (total_loss / max(seen, 1)), (correct / max(seen, 1))


def train_pipeline1() -> BinaryEwasteNet:
    set_seed(SEED)
    model = BinaryEwasteNet().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=P1_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P1_EPOCHS)
    scaler = make_scaler()

    val_loader = p1_loaders.get("val") or p1_loaders.get("test")
    best_acc, history = -1.0, []

    for epoch in range(1, P1_EPOCHS + 1):
        tr_loss, tr_acc = run_epoch_p1(model, p1_loaders["train"], criterion, optimizer, scaler)
        if val_loader is not None:
            va_loss, va_acc = run_epoch_p1(model, val_loader, criterion)
        else:
            va_loss, va_acc = float("nan"), tr_acc
        scheduler.step()
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                            val_loss=va_loss, val_acc=va_acc))
        print(f"[P1] epoch {epoch:02d}/{P1_EPOCHS}  "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save({"state_dict": model.state_dict(), "val_acc": va_acc,
                        "arch": "resnet18", "epoch": epoch}, P1_CKPT)

    print(f"[P1] best val accuracy {best_acc:.4f} -> {P1_CKPT}")
    ckpt = torch.load(P1_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    globals()["p1_history"] = pd.DataFrame(history)
    return model.eval()


def train_pipeline1_yolo_cls():
    """Alternative: ultralytics YOLOv8-cls on an ImageFolder tree.

    Expects CROP_DIR/<split>/<class>/*.jpg, which crop_yolo_export already
    produces. Returns the ultralytics model; inference below stays on the
    torch path, so treat this as a benchmarking side-door.
    """
    if not HAS_ULTRALYTICS:
        raise RuntimeError("ultralytics is not installed")
    yolo = YOLO("yolov8n-cls.pt")
    yolo.train(data=str(CROP_DIR), epochs=P1_EPOCHS, imgsz=IMG_SIZE, seed=SEED,
               project=str(CKPT_DIR), name="p1_yolo_cls", exist_ok=True)
    return yolo


p1_model = train_pipeline1_yolo_cls() if USE_YOLO_CLS else train_pipeline1()


# ---------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------
@torch.inference_mode()
def predict_pipeline1(image_path: str) -> Dict[str, Any]:
    """Stage 1 gate.

    Returns is_ewaste / confidence, and pre-seeds `category` to
    "MixedPlastics" when the binary label is 0 so a plastics lot is already
    Data-Dictionary-valid without ever touching Pipeline 2.
    """
    result: Dict[str, Any] = {
        "image_path": str(image_path), "is_ewaste": None,
        "confidence": 0.0, "category": None, "error": None,
    }
    try:
        image = load_image(str(image_path))
    except FileNotFoundError as exc:
        result["error"] = str(exc)
        return result

    tensor = eval_tf(image).unsqueeze(0).to(DEVICE)
    probs = torch.softmax(p1_model(tensor), dim=1).squeeze(0).cpu().numpy()
    label = int(np.argmax(probs))

    result["is_ewaste"] = bool(label == 1)
    result["confidence"] = float(probs[label])
    result["prob_ewaste"] = float(probs[1])
    result["category"] = None if label == 1 else "MixedPlastics"  # filled by Pipeline 2 when e-waste
    return result


_probe = p1_df[p1_df["split"].isin(["test", "val"])].head(1)
if not _probe.empty:
    print("\nSample Pipeline 1 inference:")
    print(json.dumps(predict_pipeline1(_probe.iloc[0]["image_path"]), indent=2))

## Cell 4 — Pipeline 2: Hierarchical Multi-Task Classifier

One shared ResNet34 trunk feeding three heads: `subCategory` (CE, `ignore_index=-100` for
crops with no device-level label), component presence (BCE, multi-label), and metal mass
fractions (sigmoid + MSE).

In [ ]:
# =====================================================================
# CELL 4 - Pipeline 2: hierarchical multi-task classifier
# =====================================================================
# Runs only when Pipeline 1 says is_ewaste == True. One ResNet34 trunk,
# three heads:
#   (a) subCategory      - single-label, CrossEntropy, ignore_index=-100
#   (b) components       - multi-label, BCEWithLogits  (has_motor, ...)
#   (c) metal content    - 4 mass fractions, sigmoid + MSE
# Sharing a trunk is deliberate: the crops are small and the three tasks
# key off the same visual evidence, so the auxiliary heads act as
# regularisers on the subCategory head.

P2_EPOCHS = 12
P2_LR = 3e-4
P2_CKPT = CKPT_DIR / "pipeline2_multitask.pt"
LOSS_WEIGHTS = {"sub": 1.0, "comp": 0.6, "metal": 0.8}


class MultiTaskEwasteNet(nn.Module):
    def __init__(self, n_sub: int, n_comp: int, n_metal: int,
                 arch: str = "resnet34", pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        self.backbone, feat_dim = build_backbone(arch, pretrained)
        self.neck = nn.Sequential(nn.Linear(feat_dim, 512), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.sub_head = nn.Linear(512, n_sub)       # softmax over subCategory
        self.comp_head = nn.Linear(512, n_comp)     # independent sigmoids
        self.metal_head = nn.Linear(512, n_metal)   # bounded regression

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        feats = self.neck(self.backbone(x))
        return {
            "sub_logits": self.sub_head(feats),
            "comp_logits": self.comp_head(feats),
            "metal_frac": torch.sigmoid(self.metal_head(feats)),
        }


def multitask_loss(out: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor],
                   ce: nn.Module, bce: nn.Module) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Weighted sum of the three task losses.

    The subCategory term self-skips rows labelled -100 (crops whose source
    class maps to a category but not to a specific device type).
    """
    y_sub = batch["y_sub"].to(DEVICE)
    y_comp = batch["y_comp"].to(DEVICE)
    y_metal = batch["y_metal"].to(DEVICE)

    loss_sub = ce(out["sub_logits"], y_sub)
    if torch.isnan(loss_sub):  # whole batch was ignore_index
        loss_sub = torch.zeros((), device=DEVICE)
    loss_comp = bce(out["comp_logits"], y_comp)
    loss_metal = F.mse_loss(out["metal_frac"], y_metal)

    total = (LOSS_WEIGHTS["sub"] * loss_sub
             + LOSS_WEIGHTS["comp"] * loss_comp
             + LOSS_WEIGHTS["metal"] * loss_metal)
    return total, {"sub": float(loss_sub), "comp": float(loss_comp), "metal": float(loss_metal)}


def run_epoch_p2(model, loader, ce, bce, optimizer=None, scaler=None) -> Dict[str, float]:
    training = optimizer is not None
    model.train(training)
    agg = {"loss": 0.0, "sub": 0.0, "comp": 0.0, "metal": 0.0}
    sub_correct, sub_seen, seen = 0, 0, 0

    for batch in loader:
        keep = batch["ok"] > 0
        if keep.sum() == 0:
            continue
        batch = {k: v[keep] for k, v in batch.items()}
        images = batch["image"].to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                out = model(images)
                loss, parts = multitask_loss(out, batch, ce, bce)
            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        bs = images.size(0)
        seen += bs
        agg["loss"] += float(loss) * bs
        for k in ("sub", "comp", "metal"):
            agg[k] += parts[k] * bs

        y_sub = batch["y_sub"].to(DEVICE)
        valid = y_sub != -100
        if valid.any():
            preds = out["sub_logits"].argmax(1)
            sub_correct += int((preds[valid] == y_sub[valid]).sum())
            sub_seen += int(valid.sum())

    stats = {k: v / max(seen, 1) for k, v in agg.items()}
    stats["sub_acc"] = sub_correct / max(sub_seen, 1)
    return stats


def train_pipeline2() -> MultiTaskEwasteNet:
    set_seed(SEED)
    model = MultiTaskEwasteNet(
        n_sub=len(CANONICAL_SUBCATEGORIES), n_comp=len(COMPONENT_KEYS), n_metal=len(METAL_KEYS)
    ).to(DEVICE)

    ce = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.05)
    bce = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=P2_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P2_EPOCHS)
    scaler = make_scaler()

    val_loader = p2_loaders.get("val") or p2_loaders.get("test")
    best_score, history = -1.0, []

    for epoch in range(1, P2_EPOCHS + 1):
        tr = run_epoch_p2(model, p2_loaders["train"], ce, bce, optimizer, scaler)
        va = run_epoch_p2(model, val_loader, ce, bce) if val_loader else tr
        scheduler.step()
        history.append({"epoch": epoch,
                        **{f"train_{k}": v for k, v in tr.items()},
                        **{f"val_{k}": v for k, v in va.items()}})
        print(f"[P2] epoch {epoch:02d}/{P2_EPOCHS}  train loss {tr['loss']:.4f} "
              f"subAcc {tr['sub_acc']:.4f} | val loss {va['loss']:.4f} subAcc {va['sub_acc']:.4f} "
              f"(metal mse {va['metal']:.5f})")

        score = va["sub_acc"] - va["metal"]  # accuracy, penalised by composition error
        if score > best_score:
            best_score = score
            torch.save({"state_dict": model.state_dict(),
                        "subcategories": CANONICAL_SUBCATEGORIES,
                        "component_keys": COMPONENT_KEYS,
                        "metal_keys": METAL_KEYS,
                        "epoch": epoch}, P2_CKPT)

    print(f"[P2] best checkpoint -> {P2_CKPT}")
    model.load_state_dict(torch.load(P2_CKPT, map_location=DEVICE)["state_dict"])
    globals()["p2_history"] = pd.DataFrame(history)
    return model.eval()


p2_model = train_pipeline2()


# ---------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------
@torch.inference_mode()
def predict_pipeline2(image_path: str, comp_threshold: float = 0.5) -> Dict[str, Any]:
    """Stage 2: subCategory + component presence + metal composition.

    Metal fractions are renormalised if the heads jointly predict more than
    100% of the lot mass - a physical constraint the loss does not enforce.
    """
    result: Dict[str, Any] = {
        "image_path": str(image_path), "category": None, "subCategory": None,
        "subCategoryConfidence": 0.0, "components": {}, "componentProbs": {},
        "metalContent": {}, "error": None,
    }
    try:
        image = load_image(str(image_path))
    except FileNotFoundError as exc:
        result["error"] = str(exc)
        return result

    tensor = eval_tf(image).unsqueeze(0).to(DEVICE)
    out = p2_model(tensor)

    sub_probs = torch.softmax(out["sub_logits"], dim=1).squeeze(0).cpu().numpy()
    sub_idx = int(np.argmax(sub_probs))
    sub_name = CANONICAL_SUBCATEGORIES[sub_idx]

    comp_probs = torch.sigmoid(out["comp_logits"]).squeeze(0).cpu().numpy()
    metal = out["metal_frac"].squeeze(0).cpu().numpy().astype(float)
    total = float(metal.sum())
    if total > 1.0:
        metal = metal / total

    result.update({
        "category": SUBCATEGORY_TO_CATEGORY[sub_name],   # hierarchy: sub -> category
        "subCategory": sub_name,
        "subCategoryConfidence": float(sub_probs[sub_idx]),
        "subCategoryProbs": {c: float(p) for c, p in zip(CANONICAL_SUBCATEGORIES, sub_probs)},
        "components": {k: bool(p >= comp_threshold) for k, p in zip(COMPONENT_KEYS, comp_probs)},
        "componentProbs": {k: float(p) for k, p in zip(COMPONENT_KEYS, comp_probs)},
        "metalContent": {k: round(float(v), 4) for k, v in zip(METAL_KEYS, metal)},
    })
    return result


_probe2 = p2_df[p2_df["split"].isin(["test", "val"])].head(1)
if not _probe2.empty:
    print("\nSample Pipeline 2 inference:")
    print(json.dumps(predict_pipeline2(_probe2.iloc[0]["image_path"]), indent=2))

## Cell 5 — Pipeline 3: XGBoost Price Regressor

Features are Pipeline 2 **predictions** (not the priors) plus tabular metadata, so the
regressor learns to price from noisy vision output. Target is fit in log space.

Swap `simulate_estimated_value()` for a join against the `Price Dataset Entry` table once
real `buyingPrice` / `sellingPrice` rows exist — nothing else in the cell changes.

In [ ]:
# =====================================================================
# CELL 5 - Pipeline 3: price discovery (XGBoost regressor)
# =====================================================================
# Features = Pipeline 2 *predictions* (encoded category/subCategory,
# component probabilities, metal fractions) + tabular metadata
# (approxWeightKg, latitude, longitude).
# Target   = estimatedValue in INR.
#
# IMPORTANT - the target here is SIMULATED from published scrap rates and
# the composition priors, because no historical price ledger exists yet.
# Once the Price Dataset Entry table (materialCategory, location, date,
# buyingPrice, sellingPrice, unit) has real rows, replace
# simulate_estimated_value() with a join against it and retrain. Nothing
# else in this cell changes.

P3_MODEL_PATH = ARTIFACT_DIR / "pipeline3_price_xgb.json"

# Placeholder market rates, INR per kg of recovered metal (Sept 2026 order
# of magnitude). Keep these in one place; ops will want to edit them.
METAL_RATE_INR_PER_KG: Dict[str, float] = {
    "pct_copper": 620.0,
    "pct_aluminum": 160.0,
    "pct_steel_iron": 38.0,
    "pct_precious_pcb": 1400.0,
}
# Floor rate the material fetches regardless of metal content, INR/kg.
BASE_RATE_INR_PER_KG: Dict[str, float] = {
    "PCB": 180.0, "CRT": 12.0, "Cables": 210.0, "Battery": 90.0,
    "Motor": 55.0, "MixedPlastics": 18.0, "OtherEwaste": 30.0,
}
# Premium a recycler pays when a component is actually present, INR/kg.
COMPONENT_BONUS_INR_PER_KG: Dict[str, float] = {
    "has_motor": 6.0, "has_cables": 9.0, "has_pcb": 25.0,
    "has_battery": 7.0, "has_display": 4.0,
}
# Regional demand multipliers, keyed to collection hubs.
CITY_HUBS: Dict[str, Dict[str, float]] = {
    "Indore":  {"lat": 22.7196, "lon": 75.8577, "mult": 1.00},
    "Bhopal":  {"lat": 23.2599, "lon": 77.4126, "mult": 0.96},
    "Nagpur":  {"lat": 21.1458, "lon": 79.0882, "mult": 0.98},
    "Pune":    {"lat": 18.5204, "lon": 73.8567, "mult": 1.08},
}
# Typical single-lot weight by category (lognormal mu/sigma over kg).
WEIGHT_PRIORS_KG: Dict[str, Tuple[float, float]] = {
    "PCB": (-0.7, 0.5), "Cables": (0.1, 0.6), "MixedPlastics": (-0.2, 0.5),
    "OtherEwaste": (0.3, 0.7), "CRT": (2.4, 0.3), "Battery": (-0.5, 0.4), "Motor": (0.8, 0.5),
}


# ---------------------------------------------------------------------
# 5.1  Batched CV feature extraction
# ---------------------------------------------------------------------
@torch.inference_mode()
def extract_cv_features(df: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    """Run Pipeline 2 over a DataFrame and return one feature row per image.

    Features are model outputs, not the priors - so the regressor has to
    cope with the same noise it will see in production.
    """
    ds = LotImageDataset(df, eval_tf, mode="p2")
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    p2_model.eval()

    sub_pred, sub_conf, comp_rows, metal_rows, ok_rows = [], [], [], [], []
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        out = p2_model(images)
        probs = torch.softmax(out["sub_logits"], dim=1)
        conf, idx = probs.max(dim=1)
        sub_pred.extend(idx.cpu().tolist())
        sub_conf.extend(conf.cpu().tolist())
        comp_rows.append(torch.sigmoid(out["comp_logits"]).cpu().numpy())
        metals = out["metal_frac"].cpu().numpy()
        metals = metals / np.clip(metals.sum(axis=1, keepdims=True), 1.0, None)
        metal_rows.append(metals)
        ok_rows.extend(batch["ok"].cpu().tolist())

    feats = pd.DataFrame(index=df.index)
    feats["pred_subCategory"] = [CANONICAL_SUBCATEGORIES[i] for i in sub_pred]
    feats["pred_category"] = feats["pred_subCategory"].map(SUBCATEGORY_TO_CATEGORY)
    feats["subCategoryConfidence"] = sub_conf
    comp = np.concatenate(comp_rows, axis=0)
    metal = np.concatenate(metal_rows, axis=0)
    for i, k in enumerate(COMPONENT_KEYS):
        feats[f"p_{k}"] = comp[:, i]
    for i, k in enumerate(METAL_KEYS):
        feats[f"pred_{k}"] = metal[:, i]
    feats["image_ok"] = ok_rows
    return feats[np.array(ok_rows) > 0]


# ---------------------------------------------------------------------
# 5.2  Tabular metadata + simulated target
# ---------------------------------------------------------------------
def simulate_tabular_metadata(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """Stand in for the UI-entered fields (approxWeightKg, GPS) until real
    lots exist. Weight is category-conditional; GPS is scattered around the
    active collection hubs."""
    hubs = list(CITY_HUBS.items())
    pick = rng.integers(0, len(hubs), size=len(df))
    meta = pd.DataFrame(index=df.index)
    meta["hub"] = [hubs[i][0] for i in pick]
    meta["latitude"] = [hubs[i][1]["lat"] + rng.normal(0, 0.08) for i in pick]
    meta["longitude"] = [hubs[i][1]["lon"] + rng.normal(0, 0.08) for i in pick]
    mu_sigma = df["category"].map(lambda c: WEIGHT_PRIORS_KG.get(c, (0.0, 0.6)))
    meta["approxWeightKg"] = [
        float(np.clip(rng.lognormal(mu, sigma), 0.05, 200.0)) for mu, sigma in mu_sigma
    ]
    return meta


def price_per_kg(category: str, metal_fracs: Dict[str, float],
                 components: Dict[str, float], hub_mult: float = 1.0) -> float:
    """Rule-based INR/kg used to generate the training target and as the
    fallback when the regressor is unavailable."""
    metal_value = sum(metal_fracs.get(k, 0.0) * METAL_RATE_INR_PER_KG[k] for k in METAL_KEYS)
    bonus = sum(COMPONENT_BONUS_INR_PER_KG[k] * float(components.get(k, 0.0)) for k in COMPONENT_KEYS)
    base = BASE_RATE_INR_PER_KG.get(category, BASE_RATE_INR_PER_KG["OtherEwaste"])
    return (base + metal_value + bonus) * hub_mult


def simulate_estimated_value(df: pd.DataFrame, meta: pd.DataFrame,
                             rng: np.random.Generator) -> np.ndarray:
    """Target generator. Uses the TRUE priors (not model predictions) plus
    lognormal market noise, so the regressor must learn to recover value
    from imperfect vision outputs - which is the real task."""
    values = []
    for idx in df.index:
        row = df.loc[idx]
        metal_fracs = {k: float(row[k]) for k in METAL_KEYS}
        components = {k: float(row[k]) for k in COMPONENT_KEYS}
        hub_mult = CITY_HUBS[meta.loc[idx, "hub"]]["mult"]
        per_kg = price_per_kg(row["category"], metal_fracs, components, hub_mult)
        weight = float(meta.loc[idx, "approxWeightKg"])
        noise = float(rng.lognormal(mean=0.0, sigma=0.12))   # dealer-to-dealer spread
        values.append(per_kg * weight * noise)
    return np.asarray(values, dtype=float)


# ---------------------------------------------------------------------
# 5.3  Assemble the price table
# ---------------------------------------------------------------------
rng = np.random.default_rng(SEED)
cv_feats = extract_cv_features(p2_df)
price_df = p2_df.loc[cv_feats.index].join(cv_feats)
price_df = price_df.join(simulate_tabular_metadata(price_df, rng))
price_df["estimatedValue"] = simulate_estimated_value(
    price_df, price_df[["hub", "approxWeightKg"]], rng
)

FEATURE_COLUMNS: List[str] = (
    ["category_enc", "subCategory_enc", "subCategoryConfidence", "approxWeightKg", "latitude", "longitude"]
    + [f"p_{k}" for k in COMPONENT_KEYS]
    + [f"pred_{k}" for k in METAL_KEYS]
    + ["metal_value_per_kg"]
)

category_encoder = LabelEncoder().fit(LOT_CATEGORIES)
subcategory_encoder = LabelEncoder().fit(CANONICAL_SUBCATEGORIES)


def build_feature_matrix(frame: pd.DataFrame) -> pd.DataFrame:
    """Shared by training and by the live inference path in Cell 6 - one
    definition means train/serve skew cannot creep in."""
    out = pd.DataFrame(index=frame.index)
    out["category_enc"] = category_encoder.transform(frame["pred_category"])
    out["subCategory_enc"] = subcategory_encoder.transform(frame["pred_subCategory"])
    out["subCategoryConfidence"] = frame["subCategoryConfidence"].astype(float)
    out["approxWeightKg"] = frame["approxWeightKg"].astype(float)
    out["latitude"] = frame["latitude"].astype(float)
    out["longitude"] = frame["longitude"].astype(float)
    for k in COMPONENT_KEYS:
        out[f"p_{k}"] = frame[f"p_{k}"].astype(float)
    for k in METAL_KEYS:
        out[f"pred_{k}"] = frame[f"pred_{k}"].astype(float)
    # Explicit interaction: the single most predictive engineered feature.
    out["metal_value_per_kg"] = sum(
        out[f"pred_{k}"] * METAL_RATE_INR_PER_KG[k] for k in METAL_KEYS
    )
    return out[FEATURE_COLUMNS]


X_all = build_feature_matrix(price_df)
y_all = price_df["estimatedValue"].astype(float)
split_col = price_df["split"]

X_train, y_train = X_all[split_col == "train"], y_all[split_col == "train"]
X_val, y_val = X_all[split_col == "val"], y_all[split_col == "val"]
X_test, y_test = X_all[split_col == "test"], y_all[split_col == "test"]
if len(X_val) == 0 or len(X_test) == 0:  # tiny datasets may not populate all splits
    X_train, X_hold, y_train, y_hold = train_test_split(X_all, y_all, test_size=0.3, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_hold, y_hold, test_size=0.5, random_state=SEED)

print(f"Price model rows - train {len(X_train)}, val {len(X_val)}, test {len(X_test)}")


# ---------------------------------------------------------------------
# 5.4  Train
# ---------------------------------------------------------------------
# Value is right-skewed, so fit in log space and exponentiate back. This
# keeps a 5000 INR lot from dominating the loss over a 40 INR one.
def train_price_model():
    if HAS_XGBOOST:
        model = xgb.XGBRegressor(
            n_estimators=600, learning_rate=0.05, max_depth=6,
            subsample=0.85, colsample_bytree=0.85, min_child_weight=3,
            reg_lambda=1.0, objective="reg:squarederror",
            random_state=SEED, n_jobs=-1, early_stopping_rounds=40,
        )
        model.fit(X_train, np.log1p(y_train), eval_set=[(X_val, np.log1p(y_val))], verbose=False)
        print(f"[P3] XGBoost trained; best iteration {model.best_iteration}")
    else:
        from sklearn.ensemble import GradientBoostingRegressor

        print("[P3] xgboost missing - falling back to sklearn GradientBoostingRegressor")
        model = GradientBoostingRegressor(random_state=SEED, n_estimators=400, learning_rate=0.05, max_depth=4)
        model.fit(X_train, np.log1p(y_train))
    return model


price_model = train_price_model()


def predict_estimated_value(features: pd.DataFrame) -> np.ndarray:
    """Inverse of the log1p target transform, floored at zero."""
    return np.clip(np.expm1(price_model.predict(features)), 0.0, None)


_pred_val = predict_estimated_value(X_val)
print(f"[P3] val  MAE  {mean_absolute_error(y_val, _pred_val):8.2f} INR")
print(f"[P3] val  RMSE {np.sqrt(mean_squared_error(y_val, _pred_val)):8.2f} INR")
print(f"[P3] val  R2   {r2_score(y_val, _pred_val):8.4f}")

# Persist everything Cell 6 needs to reconstruct the feature row.
if HAS_XGBOOST:
    price_model.save_model(str(P3_MODEL_PATH))
if HAS_JOBLIB:
    joblib.dump(
        {"category_encoder": category_encoder, "subcategory_encoder": subcategory_encoder,
         "feature_columns": FEATURE_COLUMNS, "metal_rates": METAL_RATE_INR_PER_KG,
         "base_rates": BASE_RATE_INR_PER_KG},
        ARTIFACT_DIR / "pipeline3_preprocessing.joblib",
    )
print(f"[P3] artifacts -> {ARTIFACT_DIR}")

## Cell 6 — End-to-End Inference

`estimate_recycle_price(photoPaths, approxWeightKg, lat, lon)` chains all three stages and
returns a validated `Lot` dict with `syncStatus="pending"`. `predict_lot_price` is an alias.

In [ ]:
# =====================================================================
# CELL 6 - End-to-end inference: photos -> Lot payload
# =====================================================================
# estimate_recycle_price() is the only function the app layer should call.
# It chains P1 -> P2 -> P3 and returns a dict that drops straight into the
# Lot table: camelCase keys, doubles for money, ISO-8601 createdAt,
# syncStatus="pending" so the sync layer picks it up.

DEFAULT_LAT, DEFAULT_LON = CITY_HUBS["Indore"]["lat"], CITY_HUBS["Indore"]["lon"]


def _nearest_hub_multiplier(lat: Optional[float], lon: Optional[float]) -> float:
    """Regional multiplier for the rule-based fallback path."""
    if lat is None or lon is None:
        return 1.0
    best, best_d = 1.0, float("inf")
    for hub in CITY_HUBS.values():
        d = (hub["lat"] - lat) ** 2 + (hub["lon"] - lon) ** 2
        if d < best_d:
            best_d, best = d, hub["mult"]
    return best


def _aggregate_photo_evidence(p2_results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Fuse per-photo Pipeline 2 outputs into one view of the lot.

    subCategory is decided by summed class probability across photos rather
    than by majority vote, so one confident shot outranks two blurry ones.
    Component probabilities take the max (a component seen in any photo is
    present); metal fractions are averaged.
    """
    prob_sum = {c: 0.0 for c in CANONICAL_SUBCATEGORIES}
    comps = {k: 0.0 for k in COMPONENT_KEYS}
    metals = {k: [] for k in METAL_KEYS}

    for res in p2_results:
        for c, p in res["subCategoryProbs"].items():
            prob_sum[c] += p
        for k, p in res["componentProbs"].items():
            comps[k] = max(comps[k], p)
        for k, v in res["metalContent"].items():
            metals[k].append(v)

    sub = max(prob_sum, key=prob_sum.get)
    return {
        "subCategory": sub,
        "category": SUBCATEGORY_TO_CATEGORY[sub],
        "subCategoryConfidence": prob_sum[sub] / max(len(p2_results), 1),
        "componentProbs": comps,
        "metalContent": {k: float(np.mean(v)) if v else 0.0 for k, v in metals.items()},
    }


def estimate_recycle_price(
    photoPaths: List[str],
    approxWeightKg: float,
    lat: Optional[float] = None,
    lon: Optional[float] = None,
    return_diagnostics: bool = False,
) -> Dict[str, Any]:
    """Full inference chain for one lot.

    Args:
        photoPaths: local image paths captured by the UI camera.
        approxWeightKg: collector-entered weight in kg.
        lat / lon: device GPS at creation; falls back to the Indore hub.

    Returns:
        A Lot-shaped dict. `estimatedValue` is None only when no photo could
        be read at all - the caller should surface that as a retry prompt
        rather than shipping a zero-value lot.
    """
    created_at = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
    lot = Lot(
        id=str(uuid.uuid4()),
        category="OtherEwaste",
        approxWeightKg=float(approxWeightKg),
        photoPaths=[str(p) for p in photoPaths],
        createdAt=created_at,
        latitude=float(lat) if lat is not None else None,
        longitude=float(lon) if lon is not None else None,
        syncStatus="pending",
    )
    diagnostics: Dict[str, Any] = {"perPhoto": [], "errors": [], "fallbackUsed": False}

    if not photoPaths:
        diagnostics["errors"].append("photoPaths is empty")
        payload = validate_lot(lot.to_dict())
        return {**payload, "diagnostics": diagnostics} if return_diagnostics else payload

    # --- Stage 1: is this e-waste at all? ----------------------------
    p1_results, readable = [], []
    for path in photoPaths:
        res = predict_pipeline1(path)
        p1_results.append(res)
        if res["error"]:
            diagnostics["errors"].append(res["error"])
        else:
            readable.append(path)
    diagnostics["perPhoto"] = p1_results

    if not readable:
        diagnostics["errors"].append("no readable photos - cannot estimate")
        payload = validate_lot(lot.to_dict())
        return {**payload, "diagnostics": diagnostics} if return_diagnostics else payload

    ewaste_votes = [r for r in p1_results if r["error"] is None and r["is_ewaste"]]
    is_ewaste = len(ewaste_votes) * 2 >= len([r for r in p1_results if r["error"] is None])
    diagnostics["isEwaste"] = is_ewaste
    diagnostics["p1Confidence"] = float(np.mean([r["confidence"] for r in p1_results if r["error"] is None]))

    # --- Stage 2: only for e-waste -----------------------------------
    if is_ewaste:
        p2_results = [r for r in (predict_pipeline2(p) for p in readable) if r["error"] is None]
        if not p2_results:
            diagnostics["errors"].append("Pipeline 2 produced no usable result")
            payload = validate_lot(lot.to_dict())
            return {**payload, "diagnostics": diagnostics} if return_diagnostics else payload
        evidence = _aggregate_photo_evidence(p2_results)
    else:
        # Binary label 0 short-circuits the hierarchy: plastics carry no
        # components and no recoverable metal.
        evidence = {
            "subCategory": None, "category": "MixedPlastics", "subCategoryConfidence": 0.0,
            "componentProbs": {k: 0.0 for k in COMPONENT_KEYS},
            "metalContent": {k: 0.0 for k in METAL_KEYS},
        }
    lot.category = evidence["category"]
    lot.subCategory = evidence["subCategory"]
    diagnostics["evidence"] = evidence

    # --- Stage 3: price ----------------------------------------------
    hub_mult = _nearest_hub_multiplier(lot.latitude, lot.longitude)
    try:
        if evidence["subCategory"] is None:
            raise ValueError("no subCategory - regressor is only fitted on e-waste rows")
        frame = pd.DataFrame([{
            "pred_category": evidence["category"],
            "pred_subCategory": evidence["subCategory"],
            "subCategoryConfidence": evidence["subCategoryConfidence"],
            "approxWeightKg": float(approxWeightKg),
            "latitude": lot.latitude if lot.latitude is not None else DEFAULT_LAT,
            "longitude": lot.longitude if lot.longitude is not None else DEFAULT_LON,
            **{f"p_{k}": evidence["componentProbs"][k] for k in COMPONENT_KEYS},
            **{f"pred_{k}": evidence["metalContent"][k] for k in METAL_KEYS},
        }])
        estimated = float(predict_estimated_value(build_feature_matrix(frame))[0])
    except Exception as exc:
        # Rule-based safety net: the collector always gets a number.
        diagnostics["fallbackUsed"] = True
        diagnostics["errors"].append(f"regressor unavailable ({exc}); used rule-based rate card")
        estimated = price_per_kg(
            evidence["category"], evidence["metalContent"], evidence["componentProbs"], hub_mult
        ) * float(approxWeightKg)

    lot.estimatedValue = round(float(estimated), 2)
    payload = validate_lot(lot.to_dict())
    return {**payload, "diagnostics": diagnostics} if return_diagnostics else payload


# Alias matching the spec's shorthand name.
predict_lot_price = estimate_recycle_price


# --- Smoke test -------------------------------------------------------
_demo_paths = p2_df[p2_df["split"].isin(["test", "val"])]["image_path"].head(2).tolist()
if _demo_paths:
    print("estimate_recycle_price() on a real lot:")
    print(json.dumps(estimate_recycle_price(_demo_paths, approxWeightKg=2.4,
                                            lat=22.7196, lon=75.8577), indent=2))

print("\nMissing-file handling:")
print(json.dumps(estimate_recycle_price(["/does/not/exist.jpg"], approxWeightKg=1.0,
                                        return_diagnostics=True)["diagnostics"]["errors"], indent=2))

## Cell 7 — Evaluation & Metrics

In [ ]:
# =====================================================================
# CELL 7 - Evaluation: confusion matrices, reports, price error plots
# =====================================================================


def plot_confusion(cm: np.ndarray, labels: List[str], title: str, ax) -> None:
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    thresh = cm.max() / 2 if cm.max() else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > thresh else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)


# ---------------------------------------------------------------------
# 7.1  Pipeline 1 - binary
# ---------------------------------------------------------------------
@torch.inference_mode()
def evaluate_pipeline1(split: str = "test") -> Tuple[np.ndarray, np.ndarray]:
    loader = p1_loaders.get(split) or p1_loaders.get("val")
    y_true, y_pred = [], []
    p1_model.eval()
    for batch in loader:
        keep = batch["ok"] > 0
        if keep.sum() == 0:
            continue
        logits = p1_model(batch["image"][keep].to(DEVICE))
        y_pred.extend(logits.argmax(1).cpu().tolist())
        y_true.extend(batch["label"][keep].tolist())
    return np.array(y_true), np.array(y_pred)


y1_true, y1_pred = evaluate_pipeline1()
print("=" * 62)
print("PIPELINE 1 - E-waste vs MixedPlastics")
print("=" * 62)
print(classification_report(y1_true, y1_pred, target_names=["MixedPlastics", "Ewaste"],
                            digits=4, zero_division=0))


# ---------------------------------------------------------------------
# 7.2  Pipeline 2 - subCategory, components, metal content
# ---------------------------------------------------------------------
@torch.inference_mode()
def evaluate_pipeline2(split: str = "test") -> Dict[str, Any]:
    loader = p2_loaders.get(split) or p2_loaders.get("val")
    sub_true, sub_pred = [], []
    comp_true, comp_prob, metal_true, metal_pred = [], [], [], []
    p2_model.eval()

    for batch in loader:
        keep = batch["ok"] > 0
        if keep.sum() == 0:
            continue
        out = p2_model(batch["image"][keep].to(DEVICE))
        y_sub = batch["y_sub"][keep]
        valid = y_sub != -100
        if valid.any():
            sub_true.extend(y_sub[valid].tolist())
            sub_pred.extend(out["sub_logits"].argmax(1).cpu()[valid].tolist())
        comp_true.append(batch["y_comp"][keep].numpy())
        comp_prob.append(torch.sigmoid(out["comp_logits"]).cpu().numpy())
        metal_true.append(batch["y_metal"][keep].numpy())
        metal_pred.append(out["metal_frac"].cpu().numpy())

    return {
        "sub_true": np.array(sub_true), "sub_pred": np.array(sub_pred),
        "comp_true": np.concatenate(comp_true), "comp_prob": np.concatenate(comp_prob),
        "metal_true": np.concatenate(metal_true), "metal_pred": np.concatenate(metal_pred),
    }


p2_eval = evaluate_pipeline2()
print("=" * 62)
print("PIPELINE 2a - subCategory")
print("=" * 62)
present = sorted(set(p2_eval["sub_true"].tolist()) | set(p2_eval["sub_pred"].tolist()))
if present:
    print(classification_report(
        p2_eval["sub_true"], p2_eval["sub_pred"], labels=present,
        target_names=[CANONICAL_SUBCATEGORIES[i] for i in present], digits=4, zero_division=0))

print("=" * 62)
print("PIPELINE 2b - component presence (weak labels -> F1 @ 0.5)")
print("=" * 62)
comp_bin_true = (p2_eval["comp_true"] >= 0.5).astype(int)
comp_bin_pred = (p2_eval["comp_prob"] >= 0.5).astype(int)
for i, key in enumerate(COMPONENT_KEYS):
    f1 = f1_score(comp_bin_true[:, i], comp_bin_pred[:, i], zero_division=0)
    print(f"  {key:<14} F1 {f1:.4f}   prevalence {comp_bin_true[:, i].mean():.3f}")
print(f"  macro F1 {f1_score(comp_bin_true, comp_bin_pred, average='macro', zero_division=0):.4f}")

print("=" * 62)
print("PIPELINE 2c - metal content (mass fraction MAE)")
print("=" * 62)
for i, key in enumerate(METAL_KEYS):
    mae = mean_absolute_error(p2_eval["metal_true"][:, i], p2_eval["metal_pred"][:, i])
    print(f"  {key:<18} MAE {mae:.4f}")


# ---------------------------------------------------------------------
# 7.3  Pipeline 3 - price
# ---------------------------------------------------------------------
y3_pred = predict_estimated_value(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, y3_pred)))
mae = float(mean_absolute_error(y_test, y3_pred))
mape = float(np.mean(np.abs((y_test - y3_pred) / np.clip(y_test, 1e-6, None))) * 100)
print("=" * 62)
print("PIPELINE 3 - estimatedValue (INR)")
print("=" * 62)
print(f"  RMSE {rmse:10.2f}")
print(f"  MAE  {mae:10.2f}")
print(f"  MAPE {mape:10.2f} %")
print(f"  R2   {r2_score(y_test, y3_pred):10.4f}")


# ---------------------------------------------------------------------
# 7.4  Figures
# ---------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

plot_confusion(confusion_matrix(y1_true, y1_pred), ["MixedPlastics", "Ewaste"],
               "P1 confusion matrix", axes[0, 0])

if present:
    plot_confusion(confusion_matrix(p2_eval["sub_true"], p2_eval["sub_pred"], labels=present),
                   [CANONICAL_SUBCATEGORIES[i] for i in present],
                   "P2 subCategory confusion matrix", axes[0, 1])
else:
    axes[0, 1].axis("off")

axes[0, 2].scatter(y_test, y3_pred, s=14, alpha=0.5)
lim = [0, float(max(y_test.max(), y3_pred.max())) * 1.05]
axes[0, 2].plot(lim, lim, "r--", lw=1)
axes[0, 2].set(xlabel="actual INR", ylabel="predicted INR",
               title=f"P3 predicted vs actual (RMSE {rmse:.0f})", xlim=lim, ylim=lim)

residuals = np.asarray(y3_pred) - np.asarray(y_test)
axes[1, 0].hist(residuals, bins=40, color="#4477aa")
axes[1, 0].axvline(0, color="r", ls="--", lw=1)
axes[1, 0].set(xlabel="predicted - actual (INR)", ylabel="lots", title="P3 residuals")

# Error grows with lot value; plotting against weight shows where.
axes[1, 1].scatter(price_df.loc[X_test.index, "approxWeightKg"], np.abs(residuals), s=14, alpha=0.5)
axes[1, 1].set(xlabel="approxWeightKg", ylabel="|error| INR", title="P3 absolute error vs weight")

importance = None
if HAS_XGBOOST and hasattr(price_model, "feature_importances_"):
    importance = pd.Series(price_model.feature_importances_, index=FEATURE_COLUMNS).sort_values()
elif hasattr(price_model, "feature_importances_"):
    importance = pd.Series(price_model.feature_importances_, index=FEATURE_COLUMNS).sort_values()
if importance is not None:
    axes[1, 2].barh(importance.index, importance.values, color="#66aa77")
    axes[1, 2].set_title("P3 feature importance")
    axes[1, 2].tick_params(axis="y", labelsize=8)
else:
    axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

# Training curves
if "p1_history" in globals() and "p2_history" in globals():
    fig2, ax2 = plt.subplots(1, 2, figsize=(12, 4))
    ax2[0].plot(p1_history["epoch"], p1_history["train_acc"], label="train")
    ax2[0].plot(p1_history["epoch"], p1_history["val_acc"], label="val")
    ax2[0].set(xlabel="epoch", ylabel="accuracy", title="P1 accuracy")
    ax2[0].legend()
    ax2[1].plot(p2_history["epoch"], p2_history["train_sub_acc"], label="train subAcc")
    ax2[1].plot(p2_history["epoch"], p2_history["val_sub_acc"], label="val subAcc")
    ax2[1].set(xlabel="epoch", ylabel="accuracy", title="P2 subCategory accuracy")
    ax2[1].legend()
    plt.tight_layout()
    plt.show()

# Machine-readable run summary for CI / experiment tracking.
run_summary = {
    "seed": SEED, "device": str(DEVICE), "demo_mode": bool(DEMO_MODE),
    "p1_rows": int(len(p1_df)), "p2_rows": int(len(p2_df)),
    "p1_macro_f1": float(f1_score(y1_true, y1_pred, average="macro", zero_division=0)),
    "p2_sub_macro_f1": float(f1_score(p2_eval["sub_true"], p2_eval["sub_pred"],
                                      average="macro", zero_division=0)) if present else None,
    "p3_rmse_inr": rmse, "p3_mae_inr": mae, "p3_mape_pct": mape,
}
(ARTIFACT_DIR / "run_summary.json").write_text(json.dumps(run_summary, indent=2))
print("\nRun summary:")
print(json.dumps(run_summary, indent=2))
if DEMO_MODE:
    print("\n*** DEMO MODE was active - these numbers describe synthetic noise images. ***")

## Where this is thin

- **`CRT`, `Battery`, `Motor` are unreachable.** Add labelled sources and a row in
  `PIPELINE_TO_LOT` for each.
- **The weak labels cap Pipeline 2.** The component and metal heads can only learn the
  per-subCategory constants in `COMPONENT_PRIORS` / `METAL_PRIORS` — the honest read of their
  metrics is "did it recover the subCategory", nothing more. Real value needs either
  component bounding boxes or recycler-reported composition fed back per lot.
- **Pipeline 3's ceiling is its own simulator.** A high R² here means the regressor inverted
  `price_per_kg()`, not that it prices scrap well.
- **`dataset_pipeline.py` ships a live-looking Roboflow API key** as the default in
  `os.environ.get("ROBOFLOW_API_KEY", ...)`. Rotate it and drop the fallback.
- **Class imbalance is handled by resampling only.** If the Roboflow sources stay this
  skewed, focal loss or per-class thresholds will beat `WeightedRandomSampler`.
- **No calibration.** `confidence` is raw softmax. Before the UI shows a ₹ figure to a
  collector, temperature-scale Pipeline 1 on the validation split and set a confidence floor
  below which the app asks for another photo instead of quoting.